In [ ]:
import rosbag
import numpy as np
import open3d as o3d
from sensor_msgs import point_cloud2 as pc2

bag_path = "/opt/ros_ws/zed_capture.bag"
topic = "/zed2/zed_node/point_cloud/cloud_registered"
topic_rgb = "/zed2/zed_node/left/image_rect_color"

with rosbag.Bag(bag_path, "r") as bag:
    _, msg, _ = next(bag.read_messages(topics=[topic]))
    # Verifica se existe cor
    has_rgb = any(f.name in ("rgb", "rgba") for f in msg.fields)
    
    print("PointCloud has RGB:", msg.fields)

    if has_rgb:
        # Leia tudo de uma vez, mantendo alinhamento
        data = np.asarray(list(pc2.read_points(
            msg,
            field_names=("x", "y", "z", "rgb"),
            skip_nans=True
        )))

        pts = data[:, :3]

        rgb = data[:, 3].astype(np.uint32)
        rgb_bytes = rgb.view(np.uint8).reshape(-1, 4)
        cols = rgb_bytes[:, :3] / 255.0

    else:
        pts = np.asarray(list(pc2.read_points(
            msg,
            field_names=("x", "y", "z"),
            skip_nans=True
        )))
        cols = None

# -------- FILTRO DE DISTÂNCIA --------
max_dist = 1.2
dists = np.linalg.norm(pts, axis=1)
mask = dists < max_dist

pts = pts[mask]
if cols is not None:
    cols = cols[mask]
# -------------------------------------

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)

if cols is not None:
    pcd.colors = o3d.utility.Vector3dVector(cols)

o3d.visualization.draw_geometries([pcd])


AttributeError: '_sensor_msgs__Image' object has no attribute 'fields'

In [ ]:
import open3d as o3d
print(o3d.__version__)
